# VocEd Lab 04 — Pixels as Vectors: k-NN Segmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/04_pixel_classifiers.ipynb)

In Labs 01–03 we converted each image to grayscale and applied a threshold.  We threw away
two-thirds of the available information (the G and B channels) and replaced the classifier
with a hard rule.

In this lab we take a completely different view: **every pixel is a point in 3-D colour
space** (R, G, B).  We flatten all training images into a list of pixels, attach their
ground-truth labels, and train a **k-Nearest Neighbours (k-NN) classifier** on them.  For
each test pixel the classifier finds its 5 nearest neighbours in the training data and
takes a majority vote.

By the end of this lab you will be able to:
- Explain what it means for a pixel to be a point in 3-D colour space.
- Flatten a batch of images into a pixel feature matrix.
- Train a `KNeighborsClassifier` on pixel colours and predict segmentation masks.
- Compare k-NN performance to thresholding in the cumulative Dice table.
- Articulate why k-NN ignores spatial context and why that matters.

## 0. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])   # (200, 3, 256, 256)
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])   # (200, 256, 256)

# Same reproducible split as all previous labs
np.random.seed(42)
idx       = np.random.permutation(N)
train_idx = idx[:160]
test_idx  = idx[160:]

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
print(f'Loaded {N} images.  Train: {len(train_idx)}  Test: {len(test_idx)}')

## 2. Visualising Colour Space

Before training a classifier, let's see whether the three classes actually cluster in RGB
colour space.  We sample a random subset of pixels from image 7 and plot them in a 2-D
projection (R vs G), coloured by their ground-truth label.

In [ ]:
IDX = 7
img7  = X[IDX]       # (3, 256, 256)
mask7 = y[IDX]       # (256, 256)

# Flatten to (H*W, 3) pixel matrix
# transpose(1,2,0) changes (3,256,256) → (256,256,3)  [channels last]
# reshape(-1, 3) collapses the spatial dims → (65536, 3)
pixels7 = img7.transpose(1, 2, 0).reshape(-1, 3)   # (65536, 3)
labels7 = mask7.reshape(-1)                         # (65536,)

# Sample 3000 pixels per class to keep the scatter plot readable
rng = np.random.default_rng(0)
colors = ['black', 'steelblue', 'crimson']
names  = ['background', 'cytoplasm', 'nucleus']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for cls in [0, 1, 2]:
    cls_pixels = pixels7[labels7 == cls]
    # rng.choice picks a random subset without replacement
    sample_n = min(3000, len(cls_pixels))
    chosen   = rng.choice(len(cls_pixels), sample_n, replace=False)
    s = cls_pixels[chosen]
    axes[0].scatter(s[:, 0], s[:, 1], c=colors[cls], s=1, alpha=0.5, label=names[cls])
    axes[1].scatter(s[:, 0], s[:, 2], c=colors[cls], s=1, alpha=0.5, label=names[cls])

axes[0].set_xlabel('Red channel');  axes[0].set_ylabel('Green channel')
axes[0].set_title('Colour space — R vs G');  axes[0].legend(markerscale=5)
axes[1].set_xlabel('Red channel');  axes[1].set_ylabel('Blue channel')
axes[1].set_title('Colour space — R vs B');  axes[1].legend(markerscale=5)
plt.tight_layout()
plt.show()

print('Do the three classes form separable clusters?  Look at the scatter above.')

## 3. Flattening the Training Images

To train the k-NN classifier we need all training pixels in a single matrix:
- **Feature matrix** `X_train_px`: shape `(total_train_pixels, 3)` — each row is one pixel's RGB values
- **Label vector** `y_train_px`: shape `(total_train_pixels,)` — each entry is `0`, `1`, or `2`

With 160 training images, each 256×256, the total pixel count is 160 × 256 × 256 = **10,485,760**.
That's a big matrix!  Training k-NN on all of them would be slow, so we **subsample** —
randomly selecting a fixed number of pixels per image.

In [ ]:
# ── Subsample pixels from each training image ─────────────────────────────────
PIXELS_PER_IMAGE = 500   # keep 500 random pixels per training image
rng = np.random.default_rng(42)

px_list  = []   # will hold pixel RGB vectors
lbl_list = []   # will hold corresponding class labels

for i in train_idx:
    # (3, 256, 256) → (256, 256, 3) → (65536, 3)
    pixels = X[i].transpose(1, 2, 0).reshape(-1, 3)
    labels = y[i].reshape(-1)         # (65536,)

    # Pick PIXELS_PER_IMAGE random pixel positions
    chosen = rng.choice(len(pixels), PIXELS_PER_IMAGE, replace=False)
    px_list.append(pixels[chosen])
    lbl_list.append(labels[chosen])

# Stack all per-image arrays into one big matrix
X_train_px = np.vstack(px_list)    # (160 * 500, 3) = (80000, 3)
y_train_px = np.hstack(lbl_list)   # (80000,)

print(f'Training pixel matrix : {X_train_px.shape}  dtype: {X_train_px.dtype}')
print(f'Training label vector : {y_train_px.shape}')

# Class balance check
for cls, name in enumerate(['background', 'cytoplasm', 'nucleus']):
    n = (y_train_px == cls).sum()
    print(f'  {name:>12s}: {n:6d}  ({100*n/len(y_train_px):.1f}%)')

## 4. Train the k-NN Classifier

`KNeighborsClassifier` stores the entire training set.  At prediction time, for each query
pixel it:
1. Computes the Euclidean distance to every stored pixel in 3-D colour space.
2. Identifies the `n_neighbors` closest stored pixels.
3. Returns the majority class label among those neighbours.

No learning happens during `fit` — the training data *is* the model.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# n_neighbors=5: each prediction is a majority vote among the 5 nearest training pixels
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)   # n_jobs=-1 uses all CPU cores

print('Fitting k-NN classifier...')
knn.fit(X_train_px, y_train_px)
print('Done.')

## 4b. Choosing k via Cross-Validation

`n_neighbors` is the only hyperparameter in k-NN, and it controls a bias–variance trade-off:

| k | Behaviour | Risk |
|---|---|---|
| Too small (k = 1) | Decided by the single nearest training pixel | High variance — one mislabelled training pixel spoils all nearby predictions |
| Moderate (k = 5–15) | Majority vote over a small colour neighbourhood | Good balance for most problems |
| Too large (k > 50) | Vote drawn from a very wide region of colour space | High bias — fine boundaries are blurred away |

Instead of guessing, we use **k-fold cross-validation**: split the training pixels into 5
equal *folds*, train on 4 folds, evaluate on the held-out fold, and repeat so every fold
serves as the test set once.  The mean accuracy over 5 folds is a reliable estimate of
how well each value of k generalises to unseen pixels.  We pick the k that maximises it.

In [ ]:
from sklearn.model_selection import cross_val_score

k_values = [1, 3, 5, 7, 11, 15, 21, 31]
cv_means  = []
cv_stds   = []

print('Sweeping k values (5-fold CV on training pixels)...')
for k in k_values:
    clf    = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    scores = cross_val_score(clf, X_train_px, y_train_px, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f'  k={k:3d}  accuracy = {scores.mean():.4f} ± {scores.std():.4f}')

best_k = k_values[int(np.argmax(cv_means))]
print(f'\nBest k = {best_k}  (CV accuracy = {max(cv_means):.4f})')

# Plot accuracy vs k
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(k_values, cv_means, yerr=cv_stds, marker='o', capsize=4, color='steelblue', label='CV accuracy ± 1 std')
ax.axvline(best_k, linestyle='--', color='crimson', label=f'best k = {best_k}')
ax.set_xlabel('k  (n_neighbors)')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('Choosing k via Cross-Validation')
ax.legend()
plt.tight_layout()
plt.show()

# Retrain the final model with the best k so all subsequent cells use it
print(f'\nRetraining k-NN with k = {best_k}...')
knn = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn.fit(X_train_px, y_train_px)
print('Done.  knn is now trained with the cross-validated best k.')

## 5. Predicting Masks for Test Images

In [ ]:
def predict_mask_knn(img, classifier):
    """
    Predict a segmentation mask for a (3, H, W) image using a pixel classifier.
    Returns a (H, W) integer mask.
    """
    H, W = img.shape[1], img.shape[2]
    # Flatten to (H*W, 3) so we can pass all pixels to the classifier at once
    pixels = img.transpose(1, 2, 0).reshape(-1, 3)   # (65536, 3)
    pred_flat = classifier.predict(pixels)            # (65536,)
    return pred_flat.reshape(H, W)                    # restore spatial shape


# ── Visualise on one test image ───────────────────────────────────────────────
sample = test_idx[0]
print(f'Running k-NN on test image {sample}...')
pred_knn = predict_mask_knn(X[sample], knn)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(X[sample].transpose(1, 2, 0))
axes[0].set_title(f'Image {sample} — RGB');  axes[0].axis('off')

axes[1].imshow(y[sample], cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[1].set_title('Ground truth');  axes[1].axis('off')

axes[2].imshow(pred_knn, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
axes[2].set_title('k-NN prediction');  axes[2].axis('off')

fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

## 6. Test-Set Dice & Cumulative Comparison

In [ ]:
def dice_score(pred, target, cls):
    pred_mask   = (pred   == cls)
    target_mask = (target == cls)
    intersection = (pred_mask & target_mask).sum()
    denom = pred_mask.sum() + target_mask.sum()
    return 1.0 if denom == 0 else 2 * intersection / denom


print('Running k-NN inference on all 40 test images (may take a minute)...')
knn_scores = []
for i in test_idx:
    pred = predict_mask_knn(X[i], knn)
    d = (dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2
    knn_scores.append(d)

mean_knn = np.mean(knn_scores)
print(f'k-NN test Dice: {mean_knn:.4f}  ±  {np.std(knn_scores):.4f}')

# ── Reference values from earlier labs (re-compute quickly) ──────────────────
def segment_plain(img, t_nucleus=0.45, t_background=0.85):
    gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]    = 1
    return pred

d_lab01 = np.mean([
    (dice_score(segment_plain(X[i]), y[i], 1) + dice_score(segment_plain(X[i]), y[i], 2)) / 2
    for i in test_idx
])

print('\nCumulative Dice comparison (test set):')
print('=' * 50)
print(f'{"Lab 01 — hand-picked thresholds":<35}  {d_lab01:.4f}')
print(f'{"Lab 04 — k-NN (colour only)":<35}  {mean_knn:.4f}')
print('=' * 50)

In [ ]:
def r2_identity(yp, gt):
    """R² vs y = x: 1 means perfect agreement with the identity line."""
    ss_res = np.sum((yp - gt) ** 2)
    ss_tot = np.sum((gt - gt.mean()) ** 2)
    return 1 - ss_res / ss_tot

def nc_ratio(mask):
    nuc = (mask == 2).sum()
    cyt = (mask == 1).sum()
    return nuc / cyt if cyt > 0 else np.nan

# ── Ground-truth and Lab 01 baseline N/C ratios ───────────────────────
nc_true  = np.array([nc_ratio(y[i]) for i in test_idx])
nc_lab01 = np.array([nc_ratio(segment_plain(X[i])) for i in test_idx])

# ── k-NN: run inference on all test images ─────────────────────────────
print('Computing k-NN N/C ratios on test set (may take a minute)...')
nc_knn = np.array([nc_ratio(predict_mask_knn(X[i], knn)) for i in test_idx])

# ── Scatter plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, nc_pred, label, color in zip(
        axes,
        [nc_lab01, nc_knn],
        ['Lab 01 — hand-picked', 'Lab 04 — k-NN (colour only)'],
        ['steelblue', 'seagreen']):

    valid = np.isfinite(nc_true) & np.isfinite(nc_pred)
    x, yp = nc_true[valid], nc_pred[valid]
    r2 = r2_identity(yp, x)

    ax.scatter(x, yp, alpha=0.6, color=color, edgecolors='none', s=40)
    lim = max(x.max(), yp.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.5, label='y = x (perfect)')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal')
    ax.set_xlabel('True N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{label}\nR² = {r2:.3f}')
    ax.legend(fontsize=9)

plt.suptitle('N/C Ratio: Predicted vs True (test set)', fontsize=13)
plt.tight_layout()
plt.show()


## 7. Where k-NN Fails: Spatial Blindness

k-NN makes decisions based **only on pixel colour**.  It has no idea whether a pixel is
near the centre of the image, at a cell boundary, or isolated in white space.  This means:
- Pixels with ambiguous colours (on the border between classes) are decided purely by which
  training pixels happen to be closest in colour space.
- Tiny "speckle" artefacts appear because a lone pixel can be assigned a different class
  from all its neighbours if its colour happens to be closer to a different class in the
  training data.

The cell below shows error maps — pixels where the k-NN was wrong — to make this concrete.

In [ ]:
# ── Error map: show where k-NN gets each class wrong ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
samples = test_idx[:2]

for row, idx in enumerate(samples):
    pred = predict_mask_knn(X[idx], knn)
    correct = (pred == y[idx])   # boolean: True where prediction is correct
    error   = ~correct           # True where prediction is WRONG

    axes[row, 0].imshow(X[idx].transpose(1, 2, 0))
    axes[row, 0].set_title(f'Image {idx} — RGB');  axes[row, 0].axis('off')

    axes[row, 1].imshow(pred, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 1].set_title('k-NN prediction');  axes[row, 1].axis('off')

    axes[row, 2].imshow(error, cmap='Reds', interpolation='nearest')
    axes[row, 2].set_title(f'Errors (red)  —  {error.mean()*100:.1f}% wrong')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()

print('Look at the error map: are errors clustered at boundaries, or scattered everywhere?')

## Wrap-up

**Key takeaways:**
- Using all three colour channels (R, G, B) rather than just grayscale gives the classifier
  richer information, which can improve segmentation.
- k-NN is easy to implement and requires no parameter tuning beyond `k`, but it is slow at
  prediction time and ignores spatial context entirely.
- Errors tend to cluster at **class boundaries** — where the colours of two classes
  overlap — confirming that colour alone is not sufficient.
- In the next lab we introduce **convolutions**, which look at a *patch* of neighbouring
  pixels rather than a single pixel, adding spatial context to the decision.

---

## Group Exercise — Varying k

The number of neighbours `k` is a hyperparameter.  Too small (k=1) and the classifier is
noisy; too large and it over-smooths boundaries.

| Person | Value to test |
|---|---|
| A | `n_neighbors=1` |
| B | `n_neighbors=10` |
| C | `n_neighbors=25` |

For each value, retrain the classifier and compute the Dice score on `test_idx[0:5]`
(5 test images only, to keep it fast).  Plot the error map for one image.

Share results and discuss:
- Which value of k gives the best Dice?  Why?
- Does the error map look more or less "speckly" as k increases?